# ResNet Classifier - Make, Model, Year
Combining VMMRdb and StanfordCars into a single dataset to train on.

### Hyperparameters

In [1]:
# Num of images per class
MIN_CLASSES = 40
MAX_CLASSES = 100

# batch size
batch_size = 68

# Number of iterations over dataset
epochs = 60

## Imports

In [2]:
import torchvision.transforms as tt
from torch.utils.data import Dataset
from torch.utils.data import ConcatDataset
from torch.utils.data import DataLoader
from torch.utils.data import Subset
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import torch

from collections import Counter, defaultdict
import scipy.io as sio
import pathlib
import os
import re
from PIL import Image
from tqdm import tqdm
import kagglehub
import random

## StanfordCars Setup

### Download Dataset

In [3]:
import kagglehub

# Download latest version
path_stanford_cars = kagglehub.dataset_download("eduardo4jesus/stanford-cars-dataset")

print("Path to dataset files:", path_stanford_cars)

100%|██████████| 1.82G/1.82G [00:45<00:00, 42.7MB/s]

Extracting files...


Path to dataset files: /home/msoliviawatt/.cache/kagglehub/datasets/eduardo4jesus/stanford-cars-dataset/versions/1


### StanfordCars Dataset Definition
Class Definition modified from PyTorch's implementation

In [4]:
import pathlib
import scipy.io as sio
from torchvision.datasets.folder import default_loader

class StanfordCars(Dataset):
    def __init__(self, root, global_class_to_idx, transform=None, max_images_per_class=None, class_normalizer=None):
        self.root = pathlib.Path(root)
        self.class_to_idx = global_class_to_idx
        self.loader = default_loader
        self.transform = transform
        self.class_normalizer = class_normalizer
        self.class_counts = Counter()

        devkit = self.root / "car_devkit/devkit"
        ann_path = devkit / "cars_train_annos.mat"
        img_dir = self.root / "cars_train/cars_train"

        annotations = sio.loadmat(ann_path, squeeze_me=True)["annotations"]
        raw_classes = sio.loadmat(devkit / "cars_meta.mat", squeeze_me=True)["class_names"].tolist()

        self.samples = []
        for ann in annotations:
            path = str(img_dir / ann["fname"])
            cls = raw_classes[ann["class"] - 1]

            if class_normalizer:
                cls = class_normalizer(cls)

            idx = self.class_to_idx.get(cls, None)
            if idx is None:
              continue

            if max_images_per_class and self.class_counts[cls] >= max_images_per_class:
              continue

            self.samples.append((path, idx))
            self.class_counts[cls] += 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = self.loader(path)
        if self.transform:
            img = self.transform(img)
        return img, y

## VMMRdb Setup

### Dataset Download

In [5]:
path_vmmrdb = kagglehub.dataset_download("prabashwara/vmmrdb-dataset")

print("Path to dataset files:", path_vmmrdb)

100%|██████████| 11.5G/11.5G [04:56<00:00, 41.8MB/s]

Extracting files...


Path to dataset files: /home/msoliviawatt/.cache/kagglehub/datasets/prabashwara/vmmrdb-dataset/versions/1


### Custom VMMRDB Dataset Definition

In [6]:
class VMMRDB_Dataset(Dataset):
    def __init__(self, root_dir, global_class_to_idx, class_normalization, max_images_per_class=None, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = global_class_to_idx
        self.classes = []
        self.class_counts = Counter()

        # Images to class names
        for class_name in sorted(os.listdir(root_dir)):
          class_path = os.path.join(root_dir, class_name)
          if os.path.isdir(class_path):
            stripped = class_normalization(class_name)
            idx = self.class_to_idx.get(stripped, None)
            if idx is None:
              continue
            for img_name in os.listdir(class_path):
              if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                if max_images_per_class and self.class_counts[stripped] >= max_images_per_class:
                  continue

                self.image_paths.append(os.path.join(class_path, img_name))
                self.labels.append(idx)
                self.class_counts[stripped] += 1


        self.classes = self.class_to_idx.keys()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label

## Dataset Combination

### Normalization Function
This function is used to normalize the class labels in both the VMMRdb dataset and StanfordCars dataset

In [7]:
# Not part of model or make
EXTRA = {
    'sedan', 'coupe', 'convertible', 'hatchback', 'suv', 'wagon', 'van',
    'minivan', 'pickup', 'truck', 'cab', 'crew', 'extended', 'regular',
    'quad', 'club', 'cargo', 'passenger', 'sut', 'roadster', 'hybrid',
    'conv.', 'drophead', 'supercab', 'classic', 'superleggera',
    'abarth', 'activehybrid', 'series', 'electric',
}

# Normalizes classes in both datasets
def normalize_classes(car_class):
  car_class = re.sub('_', ' ', car_class).lower()
  car_class = re.sub("-", "", car_class)
  car_class_components = car_class.split(" ")
  car_class_components = [t for t in car_class_components if t not in EXTRA]
  car_class_components = ["mercedes benz" if t == "mercedesbenz" else t for t in car_class_components]
  car_class_components = ["rolls royce" if t == "rollsroyce" else t for t in car_class_components]
  return " ".join(car_class_components)

### Combining the two datasets
Using a minimum number of 40 images and a maximum number of 100 per class.

In [8]:
# Building global class_to_idx so we can combine the two datasets properly

def build_global_class_to_idx(stanford_root, vmmrdb_root, normalize_classes, min_classes):
    class_counts = Counter()


    # Stanford Cars
    devkit = pathlib.Path(stanford_root) / "car_devkit/devkit"
    ann_path = devkit / "cars_train_annos.mat"
    annotations = sio.loadmat(ann_path, squeeze_me=True)["annotations"]
    raw_classes = sio.loadmat(devkit / "cars_meta.mat", squeeze_me=True)["class_names"].tolist()

    # Normalizing class names + retrieving image count per class
    for ann in annotations:
        cls = normalize_classes(raw_classes[ann["class"] - 1])
        class_counts[cls] += 1

    # VMMRdb
    for class_name in sorted(os.listdir(vmmrdb_root)):
        class_path = os.path.join(vmmrdb_root, class_name)
        if os.path.isdir(class_path):

            # Normalizing class names + retrieving image count per class
            cls = normalize_classes(class_name)
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".jpeg", ".png")):
                    class_counts[cls] += 1

    # Removing classes with less than min_classes number of images
    trimmed_classes = {cls for cls, count in class_counts.items() if count >= min_classes}
    class_to_idx = {cls : idx for idx, cls in enumerate(sorted(trimmed_classes))}
    class_counts = {cls : count for cls,count in class_counts.items() if cls in trimmed_classes}

    return class_to_idx, class_counts

# Building the global class to idx and finding the number of images per class + removes classes with less than MIN_CLASSES images
global_class_to_idx, class_counts = build_global_class_to_idx(path_stanford_cars, path_vmmrdb, normalize_classes, MIN_CLASSES)

### Defining Training and Test Transforms

Run the following cell if MIN_CLASSES or MAX_CLASSES is modified or the dataset is modified to get the new pixel mean and std. Then replace mean and std in the cell following this one with the new mean and std.

In [9]:
transform = tt.Compose([
    tt.Resize((224,224)),
    tt.ToTensor(),
])
# Full dataset used to calculate pixel mean and std
stanford_dataset = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=transform
)
vmmrdb_dataset = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=transform
)
full_dataset = ConcatDataset([stanford_dataset, vmmrdb_dataset])
data_loader = DataLoader(full_dataset, batch_size = batch_size, shuffle=False)


# Mean and Standard Deviation Calculation
mean = 0
ex2 = 0
total_pixels = 0

loop = tqdm(data_loader)
for images, _ in loop:
    B, C, H, W = images.shape

    images = images.view(B, C, -1) # H x W

    mean += images.sum(dim=(0, 2))
    ex2 += (images ** 2).sum(dim=(0, 2))

    total_pixels += B * H * W

    loop.set_description("mean + std calculation")

mean /= total_pixels
ex2 /= total_pixels

std = torch.sqrt(ex2 - mean ** 2)

print()
print("Mean:", mean)
print("Std:", std)

mean + std calculation: 100%|██████████| 2273/2273 [06:54<00:00,  5.48it/s]


Mean: tensor([0.4369, 0.4330, 0.4285])
Std: tensor([0.2682, 0.2650, 0.2674])


In [10]:
mean = [0.4369, 0.4330, 0.4285]
std = [0.2681, 0.2649, 0.2673]

# Defines image preprocessing for the training and test datasets
train_transform = tt.Compose([
    tt.Resize((224,224)),

    # Random Transformations
    tt.RandomHorizontalFlip(),
    tt.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    tt.RandomRotation(100),

    tt.ToTensor(),
    tt.Normalize(mean, std)
])

test_transform = tt.Compose([
    tt.Resize((224,224)),
    tt.ToTensor(),
    tt.Normalize(mean, std)
])

### Defining Datasets For Training and Testing

In [11]:
# Need to create the dataset objects
# twice to properly create the test and training sets with the correct
# transforms tied to them

# Test dataset
stanford_test = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=test_transform
)
vmmrdb_test = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=test_transform
)

# Train dataset
stanford_train = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=train_transform
)
vmmrdb_train = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=train_transform
)

### Splitting the Training and Testing datasets

In [12]:
# Splitting into training and testing
full = ConcatDataset([stanford_train, vmmrdb_train])

# 70 - 10 - 20 split
train_size = int(0.70 * len(full))
val_size = int(0.10 * len(full))
test_size = len(full) - val_size - train_size

# Test + train + val set
generator = torch.Generator().manual_seed(0)
indices = torch.randperm(len(full), generator=generator)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

dataset_train = Subset(ConcatDataset([stanford_train, vmmrdb_train]), train_indices)
dataset_val = Subset(ConcatDataset([stanford_test, vmmrdb_test]), val_indices)
dataset_test = Subset(ConcatDataset([stanford_test, vmmrdb_test]), test_indices)

In [13]:
train_loader = DataLoader(dataset_train, batch_size = batch_size, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size = batch_size, shuffle=False)
test_loader = DataLoader(dataset_test, batch_size = batch_size, shuffle=False)


### ResNet-50
Using the ResNet-50 Architecture to train a model that predicts model, make and year

In [14]:
class Bottleneck(nn.Module):
  expansion = 4
  def __init__(self, in_channels, out_channels, stride=1):
    super(Bottleneck, self).__init__()

    # 1x1 conv to reduce channels
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
    self.batch_norm1 = nn.BatchNorm2d(out_channels)

    # 3x3 conv to process on reduced channels
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
    self.batch_norm2 = nn.BatchNorm2d(out_channels)

    # 1x1 conv to expand channels
    self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, stride=1, padding=0, bias=False)
    self.batch_norm3 = nn.BatchNorm2d(out_channels * self.expansion)

    # Skip Connection
    self.shortcut = nn.Sequential()

    if stride != 1 or in_channels != out_channels * self.expansion:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels * self.expansion)
      )

  def forward(self, x):

    # Channel reduction
    out = F.relu(self.batch_norm1(self.conv1(x)))
    # Process
    out = F.relu(self.batch_norm2(self.conv2(out)))
    # Channel expansion
    out = self.batch_norm3(self.conv3(out))
    # Residual connection
    out += self.shortcut(x)
    return F.relu(out)

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super(ResNet, self).__init__()

        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2,
                               padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        layers = []

        # First block
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * block.expansion

        # Remaining blocks
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        # First convolution step using 7x7
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        # ResNet Layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Fully connected portion
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x



def ResNet50(num_classes):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)

## Training Code

In [ ]:
# Defines CNN, loss and optimizer
device = "cuda" if torch.cuda.is_available() else "cpu"
net = ResNet50(num_classes=len(global_class_to_idx)).to(device)
loss_criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.9)

checkpoint_path = "./checkpoint.pth"
best_checkpoint_path = "./best_checkpoint.pth"
start_epoch = 0

# Train loss, val loss and val accuracy
train_losses = []
val_losses = []
val_accuracies = []
best_val_acc = 0.0

# Trying to load trained weights + epoch information
try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    net.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    val_accuracies = checkpoint.get("val_accuracies", [])
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming from epoch {start_epoch}")
except FileNotFoundError:
    print("No checkpoint found, starting fresh.")

# Training loop
for epoch in range(start_epoch, epochs):
  running_loss = 0
  num_batches = 0
  train_loop = tqdm(train_loader, leave=True)
  net.train()

  for X, y in train_loop:
    X = X.to(device)
    y = y.to(device)
    optimizer.zero_grad()
    output = net(X)

    loss = loss_criterion(output, y)
    running_loss += loss.item()
    num_batches += 1

    loss.backward()
    optimizer.step()

    train_loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
    train_loop.set_postfix(loss=loss.item())

  train_loss = running_loss / num_batches
  train_losses.append(train_loss)

  # Validation loss and accuracy
  net.eval()
  val_loss = 0
  val_batches = 0
  correct = 0
  total = 0

  with torch.no_grad():
    val_loop = tqdm(val_loader, leave=True)
    for X, y in val_loop:
      X = X.to(device)
      y = y.to(device)

      output = net(X)
      loss = loss_criterion(output, y)

      val_loss += loss.item()
      val_batches += 1

      prediction = output.argmax(dim=1)
      correct += (prediction == y).sum().item()
      total += y.size(0)

      val_loop.set_description(f"Validation")
      val_loop.set_postfix(loss=loss.item())

  val_loss /= val_batches
  val_acc = correct / total

  val_losses.append(val_loss)
  val_accuracies.append(val_acc)

  print(f"Epoch {epoch+1} finished. Train Loss: {train_losses[-1]}, Val Loss={val_losses[-1]}, Val Acc={val_accuracies[-1]}")

  # Save best model
  if val_acc > best_val_acc:
    best_val_acc = val_acc
    checkpoint = {
      "epoch": epoch,
      "model_state": net.state_dict(),
      "optimizer_state": optimizer.state_dict(),
      "train_losses": train_losses,
      "val_losses": val_losses,
      "val_accuracies": val_accuracies,
      "best_val_acc": best_val_acc
    }
    torch.save(checkpoint, best_checkpoint_path)

  # Save every epoch
  checkpoint = {
    "epoch": epoch,
    "model_state": net.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_losses": train_losses,
    "val_losses": val_losses,
    "val_accuracies": val_accuracies,
    "best_val_acc": best_val_acc
  }
  torch.save(checkpoint, checkpoint_path)

  


No checkpoint found, starting fresh.


Validation: 100%|██████████| 228/228 [01:01<00:00,  3.69it/s, loss=7.57]


Epoch 1 finished. Train Loss: 7.600074276543503, Val Loss=7.565762429906611, Val Acc=0.0010353306587291317


Validation: 100%|██████████| 228/228 [01:08<00:00,  3.35it/s, loss=7.42]


Epoch 2 finished. Train Loss: 7.5030083976999755, Val Loss=7.441599626290171, Val Acc=0.0015529959880936975


...

Epoch 42 finished. Train Loss: 0.6993126251051667, Val Loss=2.4359464556501624, Val Acc=0.3850135887148958


Validation: 100%|██████████| 228/228 [01:17<00:00,  2.95it/s, loss=1.49]


Epoch 43 finished. Train Loss: 0.6575589501992627, Val Loss=2.5062375936591836, Val Acc=0.3762132781156982


Epoch [44/60]:  25%|██▍       | 394/1591 [06:02<18:20,  1.09it/s, loss=0.742]


KeyboardInterrupt: 